# Hackalysis

This notebook contains detailed analysis of the hackathon repositories, including metadata and concept extraction. The goal is to understand the lifecycle of project contributions in relation to hackathon events. Also, to be able to detect Hackathon repositories through various signals

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd

load_dotenv()

DATA_ROOT = Path(os.getenv('DATA_ROOT'))
PROVIDER_PREFIX = 'lauzhack'

## Project Based Analysis

### Create Project Based Analysis Data set 

In this data set there is 1 row per project and repo meta data is appended as a list of dicts. This allows us to do project based analysis and also to have all the repos of a project together.

In [7]:
def _load_hackathon_metadata_row(metadata_json_path: Path, year: int) -> dict:
    metadata = json.loads(metadata_json_path.read_text(encoding='utf-8'))
    if isinstance(metadata, list):
        metadata = metadata[0] if metadata else {}
    if not isinstance(metadata, dict):
        metadata = {}

    row = {f'hackathon_{k}': v for k, v in metadata.items()}
    row['hackathon_year'] = year
    return row


def load_projects_analysis_ready(data_root: Path, provider_prefix: str = 'lauzhack') -> pd.DataFrame:
    frames: list[pd.DataFrame] = []

    for folder in sorted(data_root.glob(f'{provider_prefix}-*')):
        if not folder.is_dir():
            continue

        year_str = folder.name.split('-')[-1]
        if not year_str.isdigit():
            continue
        year = int(year_str)

        projects_path = folder / f'{provider_prefix}_projects.parquet'
        github_projects_path = folder / f'{provider_prefix}_github_project_metadata.parquet'
        metadata_json_path = folder / f'{provider_prefix}_metadata.json'

        if not projects_path.exists() or not metadata_json_path.exists():
            continue

        # Prefer project-level GitHub-enriched file when present.
        source_path = github_projects_path if github_projects_path.exists() else projects_path
        df = pd.read_parquet(source_path).copy()
        df['year'] = year

        metadata_row = _load_hackathon_metadata_row(metadata_json_path, year)
        for col, val in metadata_row.items():
            if isinstance(val, (list, dict, tuple, set)):
                df[col] = [val for _ in range(len(df))]
            else:
                df[col] = val

        frames.append(df)

    if not frames:
        return pd.DataFrame()

    merged = pd.concat(frames, ignore_index=True)

    # Harmonize expected GitHub columns even when some years have no github_project_metadata file.
    if 'github_repo_urls' not in merged.columns:
        merged['github_repo_urls'] = [[] for _ in range(len(merged))]
    if 'github_repos_metadata' not in merged.columns:
        merged['github_repos_metadata'] = [[] for _ in range(len(merged))]
    if 'github_repo_count' not in merged.columns:
        merged['github_repo_count'] = 0

    merged['github_repo_count'] = pd.to_numeric(merged['github_repo_count'], errors='coerce').fillna(0).astype(int)

    # Global key for cross-year/provider uniqueness.
    if 'project_uid' in merged.columns:
        merged['global_project_uid'] = merged.apply(
            lambda r: f"{provider_prefix}:{int(r['year'])}:{r['project_uid']}" if pd.notna(r.get('project_uid')) else f"{provider_prefix}:{int(r['year'])}:row:{r.name}",
            axis=1,
        )
    else:
        merged['global_project_uid'] = merged.apply(
            lambda r: f"{provider_prefix}:{int(r['year'])}:row:{r.name}", axis=1
        )

    return merged



projects_df = load_projects_analysis_ready(DATA_ROOT, PROVIDER_PREFIX)
print(f"Loaded {len(projects_df)} projects across all years.\n rows = {len(projects_df)} \n columns = {len(projects_df.columns)}")
projects_df.columns

Loaded 213 projects across all years.
 rows = 213 
 columns = 27 


Index(['id', 'title', 'description', 'url', 'team', 'awards', 'categories',
       'hackathon_name', 'hackathon_year', 'hackathon_location', 'project_uid',
       'project_id', 'project_title', 'github_repo_urls', 'github_repo_count',
       'github_repos_metadata', 'year', 'hackathon_source_url',
       'hackathon_description', 'hackathon_date', 'hackathon_date_start',
       'hackathon_date_end', 'hackathon_social_links',
       'hackathon_extracted_at', 'tags', 'image_url', 'global_project_uid'],
      dtype='str')

#### Exploratory Data Analysis

In [ ]:
# projects_df[['title', 'description', 'team', 'awards', 'categories', 'hackathon_year', 'project_title']]

def _to_list(v):
    if isinstance(v, list):
        return v
    if pd.isna(v):
        return []
    if isinstance(v, (tuple, set)):
        return list(v)
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return []
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            return [s]
    return [v]

projects_df[['awards', 'categories']] = projects_df[['awards', 'categories']].apply(lambda col: col.apply(_to_list))

In [33]:
projects_df["awards"].equals(projects_df["categories"])

True

In [13]:
projects_df.info()

## Remove Duplicates
projects_df[projects_df.duplicated(subset=['title'], keep=False)].sort_values('title')[['year', 'title', 'github_repo_urls', 'github_repo_count', 'description', 'team', 'github_repos_metadata']]


<class 'pandas.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      213 non-null    int64 
 1   title                   213 non-null    str   
 2   description             213 non-null    str   
 3   url                     213 non-null    str   
 4   team                    213 non-null    str   
 5   awards                  97 non-null     str   
 6   categories              97 non-null     str   
 7   hackathon_name          213 non-null    str   
 8   hackathon_year          213 non-null    int64 
 9   hackathon_location      213 non-null    str   
 10  project_uid             213 non-null    str   
 11  project_id              213 non-null    str   
 12  project_title           213 non-null    str   
 13  github_repo_urls        213 non-null    str   
 14  github_repo_count       213 non-null    int64 
 15  github_repos_meta

,year,title,github_repo_urls,github_repo_count,description,team,github_repos_metadata
16,2023,Amazon Review Tools,"[""https://github.com/AliEmreSenel/LauzHack2023""]",1,Our project aims to provide an accessible inte...,"[""Ali Emre Senel"", ""Lorenzo Calda""]","[{""input_url"": ""https://github.com/AliEmreSene..."
17,2023,Amazon Review Tools,"[""https://github.com/AliEmreSenel/LauzHack2023""]",1,Our project aims to provide an accessible inte...,"[""Alberto Paolo Lolli""]","[{""input_url"": ""https://github.com/AliEmreSene..."
18,2023,Bioicons PDB2Vector,"[""https://github.com/bioicons/pdb2vector""]",1,A service to create vector illustrations from ...,"[""Simon Dürr""]","[{""input_url"": ""https://github.com/bioicons/pd..."
44,2023,Bioicons PDB2Vector,"[""https://github.com/bioicons/pdb2vector""]",1,A service to create vector illustrations from ...,"[""Ryoma Maeda"", ""Felix Richter"", ""Laurenz Rasc...","[{""input_url"": ""https://github.com/bioicons/pd..."
31,2023,Legacy LM,[],0,"A personalized conversational AI, preserving v...","[""Alejandro Hernández Cano"", ""Arvind Menon"", ""...",[]
51,2023,Legacy LM,"[""https://github.com/lars-quaedvlieg/Lauzhack-...",1,"A personalized conversational AI model, preser...","[""Somesh Mehra""]","[{""input_url"": ""https://github.com/lars-quaedv..."
35,2023,OpenLogs Lauzhack,"[""https://github.com/EncryptEx/LauzHack23""]",1,Tired of analizing log data by yourself? Try o...,"[""Jaume López Molina""]","[{""input_url"": ""https://github.com/EncryptEx/L..."
59,2023,OpenLogs Lauzhack,"[""https://github.com/EncryptEx/LauzHack23""]",1,Tired of analizing log data by yourself? Try o...,"[""Joffre Alcivar Riera"", ""Pau Carulla Lechosa""]","[{""input_url"": ""https://github.com/EncryptEx/L..."
